# 01. Ingestão e Validação Inicial dos Dados

## Contexto do Problema
O objetivo deste projeto é desenvolver um modelo de aprendizagem de máquina para auxiliar na identificação precoce de doenças cardíacas utilizando o dataset Cleveland Heart Disease (UCI Machine Learning Repository).

A detecção precoce de condições cardiovasculares permite intervenções médicas preventivas. O conjunto de dados reúne variáveis clínicas e dados demográficos de pacientes. A tarefa consiste em classificar a presença ou ausência de doença cardíaca com base nessas medições.

### Etapas do Notebook
1. Importação das bibliotecas essenciais.
2. Leitura do conjunto de dados bruto a partir da fonte oficial.
3. Inspeção inicial da estrutura, tipos de dados e valores ausentes.
4. Adequação da variável alvo para classificação binária.
5. Persistência do dado bruto tratado no diretório `data/raw/`.

In [1]:
import pandas as pd
import numpy as np

# Configuração para exibir todas as colunas no Jupyter
pd.set_option('display.max_columns', None)

## 1. Carregamento do Dataset

Os dados originais utilizam o caractere `?` para indicar entradas ausentes. A leitura do arquivo é feita definindo esse caractere como `NaN` para garantir a correta interpretação pelo Pandas.

### Dicionário de Variáveis
- **age**: Idade do paciente (em anos)
- **sex**: Sexo biológico (1 = masculino; 0 = feminino)
- **cp**: Tipo de dor no peito (1: angina típica, 2: angina atípica, 3: dor não-anginosa, 4: assintomático)
- **trestbps**: Pressão arterial em repouso (mm Hg na admissão hospitalar)
- **chol**: Colesterol sérico (mg/dl)
- **fbs**: Glicemia em jejum > 120 mg/dl (1 = verdadeiro; 0 = falso)
- **restecg**: Resultados eletrocardiográficos em repouso (0: normal, 1: alteração no segmento ST-T, 2: hipertrofia ventricular esquerda)
- **thalach**: Frequência cardíaca máxima atingida em teste de esforço
- **exang**: Angina induzida por exercício (1 = sim; 0 = não)
- **oldpeak**: Depressão do segmento ST induzida por exercício em relação ao repouso
- **slope**: Inclinação do segmento ST no pico do exercício
- **ca**: Número de vasos principais (0-3) coloridos por fluorosopia
- **thal**: Exame de cintilografia miocárdica (3 = normal; 6 = defeito fixo; 7 = defeito reversível)
- **target**: Diagnóstico de doença cardíaca

In [2]:
# Fonte dos dados brutos
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Definição dos nomes das colunas conforme documentação da UCI
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", 
    "restecg", "thalach", "exang", "oldpeak", "slope", 
    "ca", "thal", "target"
]

# Leitura do dataset
df = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?")

print(f"Dimensões do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas.\n")
df.head()

Dimensões do dataset: 303 linhas e 14 colunas.



,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2. Inspeção Técnica da Estrutura

Verificação do formato das colunas e contagem preliminar de valores ausentes por variável.

In [3]:
# Informações estruturais dos dados
df.info()

print("\n--- Quantidade de valores ausentes por coluna ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB

--- Quantidade de valores ausentes por coluna ---
ca      4
thal    2
dtype: int64


## 3. Adequação da Variável Alvo (`target`)

O dataset original possui valores de 0 a 4 na coluna `target`:
- **0**: Ausência de doença cardíaca.
- **1, 2, 3, 4**: Presença de doença cardíaca em diferentes níveis de severidade.

Para simplificar a abordagem para um problema de classificação binária, os valores maiores que 0 são convertidos para 1 (presença da doença).

In [4]:
# Binarização da variável alvo
df['target'] = (df['target'] > 0).astype(int)

# Verificação da distribuição das classes
print("Distribuição das classes (Proporção):")
print(df['target'].value_counts(normalize=True).map("{:.2%}".format))

print("\nDistribuição das classes (Absoluta):")
print(df['target'].value_counts())

Distribuição das classes (Proporção):
target
0    54.13%
1    45.87%
Name: proportion, dtype: str

Distribuição das classes (Absoluta):
target
0    164
1    139
Name: count, dtype: int64


## 4. Persistência dos Dados Brutos

Salva-se uma cópia local do DataFrame obtido no diretório `data/raw/` para garantir a reprodutibilidade dos próximos notebooks de análise exploratória e pré-processamento.

In [5]:
# Salvando o DataFrame tratado na pasta correspondente
df.to_csv("../data/raw/heart_disease_raw.csv", index=False)
print("Arquivo salvo com sucesso em: ../data/raw/heart_disease_raw.csv")

Arquivo salvo com sucesso em: ../data/raw/heart_disease_raw.csv
